# ME324 · Lab 8 — Text-gen 1/4: embeddings & character models

**Lecture 8 · "Building your own text-generating AI (1/4)" · 2026-08-12**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-08-char-bigram.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In PyTorch, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

This is the **first of four** text-generation labs:

- **Lab 8 (today):** text as data, the reusable **text pipeline**, and a **character bigram** language model. Its output will be gibberish; that's the point.
- **Lab 9:** an RNN — give the model a *memory*.
- **Lab 10:** **attention** — the move that built GPT.
- **Lab 11:** subword (BPE) tokenisation + sampling controls.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** the tokeniser (Section 2), `get_batch` (Section 3), and the bigram model, training and generation (Sections 4–5).
- **Stretch / take-home — skip if short on time:** the embedding-table peek at the end of Section 5.

_Four `# TODO` cells this time — two of them whole functions with a spec docstring. Worked answers are in the **Solutions** section at the bottom._

## Run me first

This imports PyTorch and sets `torch.manual_seed(1337)` — the nanoGPT seed — so your "random" batches and samples match your neighbour's.

> **GPU:** a bigram runs fine on CPU, but pick a GPU runtime anyway (**Runtime → Change runtime type → GPU**): Labs 9–11 will want it, and `device` below picks it up automatically.

In [ ]:
# Imports & setup  (Shift+Enter to run)
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__)
print("Using device:", device)

## Section 1 · Get the data

We'll use **tiny-shakespeare** — ~1 MB of Shakespeare, enough to train on in minutes and produce English-ish output (we'll use **the same corpus in Labs 9, 10 and 11** too.) 

`wget -nc` ("no-clobber") skips the download if `input.txt` already exists, so re-running is safe.

In [ ]:
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

Read the file into a Python string and look at it — how long, and what's in it?

In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters:", len(text))
print("-" * 60)
print(text[:500])

About **1.1 million characters** of dialogue — not "Shakespeare" to the model, just a long sequence of characters.
Since a neural network works on **numbers** and our raw data is **text**, our first job is to convert the data.

## Section 2 · A character tokeniser

**Tokenisation** is the choice of *what counts as one unit*. Today, we will use the simplest possible version, where **one character = one token**.

1. The *vocabulary* is just every **unique character** in the corpus
2. We assign an integer **id** per character (0, 1, 2, …).
3. And use two functions to move from text to numbers and back again:
    * `encode`: string → ids.  
    * `decode`: ids → string.

(**Lab 11** upgrades our tokentizer to **BPE**, the subword scheme GPT and Claude actually use.)

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("".join(chars))
print("vocab_size:", vocab_size)

There are **65** distinct characters -- newline, space, punctuation, digits, letters -- and we can use dictionaries to store a translation between strings and ids. `stoi` ("string to integer") maps each character to its id; `itos` maps back.

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
print("stoi['a'] =", stoi['a'])
print("itos[46]  =", repr(itos[46]))

Now we need a function that uses these dictionaries to return *lists* of ids. **We want `encode(s)` to return the list of ids for a string, and `decode(l)` to rebuild the string. How would you write each in one line?**

In [ ]:
# TODO: complete the two lambda bodies.
#   encode("hi")                    -> [stoi['h'], stoi['i']]
#   decode([stoi['h'], stoi['i']])  -> "hi"
encode = lambda s: None   # <-- TODO
decode = lambda l: None   # <-- TODO

Always test your code! If you encode then decode a string you should get back exactly what you started with. The cell below does exactly that — if the assert fails, the usual culprit is `decode` returning a *list* of characters rather than a joined string.

In [ ]:
print(encode("hello world"))
print(decode(encode("hello world")))

# a round-trip should return the original string unchanged
assert decode(encode("To be, or not to be")) == "To be, or not to be"
print("round-trip OK")

## Section 3 · A reusable text pipeline

Now we are ready to set up our pipeline for processing, modelling, and generating text. **Everything in this section is reused unchanged in Labs 9, 10 and 11.**

### The model interface 

Every model this week will use this pipeline:

```python
logits, loss = model(idx, targets)      # idx, targets: (B, T) Long ;  logits: (B, T, vocab)
idx = model.generate(idx, n)            # idx: (B, T)  ->  (B, T + n)
```

`idx` holds `B` sequences of `T` token ids; `forward` returns **logits** (a score for every possible next character at every position) and, given `targets`, a scalar **loss**; `generate` appends `n` tokens. Later we can swap our bigram model for a GPT and nothing else will need to change.

First, we can encode the **entire** corpus into one long 1-D tensor of token ids.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

Next, make a **train / validation split**: the first 90% trains; the last 10% is held out to catch memorisation.

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("train:", len(train_data), " val:", len(val_data))

### Blocks and batches

We won't feed the whole million characters at once (think back to Lecture 4). Instead, we'll create chunks of `block_size` characters (the *context length*), and process `batch_size` of them in parallel for speed.

In [ ]:
block_size = 8
batch_size = 32

### "Predict the next character"

Since this is generative, the training data $\mathbf{X}$ is the label -- hence this is a form of self-supervised learning. The **target** `y` is the chunk `x` shifted one right, so each 8-character block of Shakespeare can be converted into **8 prediction problems**: first character predicts second, first two predict third, and so on. Run this to see them:

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is {context.tolist()}  the target is: {target.item()}  "
          f"({decode(context.tolist())!r} -> {decode([target.item()])!r})")

### `get_batch` — drawing random training chunks

Write a function `get_batch` that picks `batch_size` random start positions, slices a `block_size` chunk at each, and builds the shifted-by-one targets. **The docstring spec below specifies the various components you need.**


In [ ]:
def get_batch(split):
    """Return one minibatch (x, y) of token ids.

    Spec:
      - d = train_data if split == 'train' else val_data
      - draw `batch_size` random START positions ix in [0, len(d) - block_size)
      - x: stack the block_size ids starting at each position  -> shape (batch_size, block_size)
      - y: the SAME slices but shifted right by one (the next char) -> shape (batch_size, block_size)
      - move x and y to `device` before returning

    Two things to find out: how torch draws random integers, and how it stacks a
    list of tensors into one. Good questions for a chatbot -- then make the
    shapes come out right.
    """
    # TODO: implement get_batch per the spec above.
    raise NotImplementedError("Implement get_batch")

Draw one batch and *look* at it — the shapes, and how each target is the next character of its input.

In [ ]:
xb, yb = get_batch('train')
print("inputs  x:", tuple(xb.shape))
print(xb)
print("targets y:", tuple(yb.shape))
print(yb)

print("\n--- unpacking the first sequence in the batch ---")
for t in range(block_size):
    context = xb[0, :t + 1].tolist()
    target = yb[0, t].item()
    print(f"when input is {context}  the target is {target}")

### `estimate_loss` — measuring progress

Averages the loss over many batches, on both splits, with gradients off (`@torch.no_grad()`).

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=200):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

### `train_model` — the training loop

Get a batch → forward for the loss → zero the gradients → backward → optimiser step — the same five steps as Lab 5. It only touches the model through the contract, so it works for *every* model in Labs 8–11; we call it in Section 5.

In [ ]:
def train_model(model, get_batch, estimate_loss, max_iters=3000, eval_interval=500,
                lr=1e-3, device='cpu'):
    """Generic training loop reused by every LM in Labs 8-11."""
    import torch
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model)
            print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model

**Pause and take stock.** These three functions *are* the pipeline; you will copy them, unchanged, into the next three labs. All that changes is the model — which we build next.

## Section 4 · The Bigram language model

A **bigram** model predicts the next character from **only the current character** — it learns $P(c_{t+1} \mid c_t)$. The neat trick (as shown in the lecture) is that the whole model is one `nn.Embedding(vocab_size, vocab_size)`. Looking up character `i` returns row `i`, which we treat directly as the **logits** for the next character. After training, row `i` *is* the count table you built by hand in the plenary, learned by gradient descent. 

**Implement `forward` and `generate` from their docstring specs.** Be aware of two issues:

- `F.cross_entropy` wants logits `(N, C)` and targets `(N,)`; ours are `(B, T, vocab)` and `(B, T)`, so flatten batch and time: `N = B*T`.
- `generate` is autoregressive: logits → keep the **last** time step → softmax → **sample** one id from those probabilities (sampling, not argmax, gives the variety) → append → repeat.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        """Spec:
          - logits = look up idx in the embedding table        -> (B, T, vocab)
          - if targets is None: return (logits, None)
          - else: flatten logits to (B*T, C) and targets to (B*T,), take
            F.cross_entropy of the two, and return (logits, loss)
        """
        # TODO: implement forward per the spec.
        raise NotImplementedError("Implement Bigram.forward")

    def generate(self, idx, max_new_tokens):
        """Spec: autoregressively sample max_new_tokens new ids.
          repeat max_new_tokens times:
            - run the model on idx; keep only the LAST time step's logits -> (B, vocab)
            - softmax them into probabilities, then draw ONE id per row from that
              distribution. torch has a function for exactly this kind of draw --
              good one to ask a chatbot: "how do I sample from a tensor of
              probabilities in PyTorch?"
            - append the sampled column to idx along the time dimension (B, T+1)
          return idx
        """
        # TODO: implement generate per the spec.
        raise NotImplementedError("Implement Bigram.generate")

Create the model and count its parameters (it should be `65 * 65 = 4225`).

In [ ]:
model = BigramLanguageModel(vocab_size).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

### A sanity check on the initial loss

**What loss should an untrained model score?** With random weights it can't beat a uniform guess over the 65 characters, whose average negative log-likelihood is

$$\text{loss} = -\ln\!\left(\tfrac{1}{\text{vocab}}\right) = \ln(\text{vocab})$$

Run the cell below and compare with the real thing. If the actual loss is far off, check the flatten in `forward` first.

In [ ]:
import math
expected = math.log(vocab_size)          # -ln(1/vocab) = ln(vocab)
print(f"expected initial loss = -ln(1/{vocab_size}) = ln({vocab_size}) = {expected:.4f}")

xb, yb = get_batch('train')
logits, loss = model(xb, yb)
print("logits shape:", tuple(logits.shape), "(expect (batch_size, block_size, vocab))")
print(f"actual initial loss   = {loss.item():.4f}")

### A sample *before* training

Generate from the untrained model, starting from a single newline character (id `0`) — pure noise, our baseline.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

## Section 5 · Train, generate, inspect

Now we can train the model, which should take well under a minute for a bigram. Watch the train and val loss fall. Increasing `max_iters` won't improve things much, because a bigram simply *cannot* do better with one character of context.

In [ ]:
train_model(model, get_batch, estimate_loss, max_iters=3000, eval_interval=500, lr=1e-3, device=device)

### Generate from the trained model

Now with trained weights — the same machinery as the before-training sample, this time 400 characters. `[0]` picks the only sequence in the batch; `.tolist()` gives `decode` the plain list it expects.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=400)[0].tolist()))

**It's gibberish** but *structured*! There are plausible letter pairs, capitalised words after line breaks, the odd real word.

With one character of context the model can't form words reliably and so in Lab 9 we'll give it a **memory**, and the output will get markedly less terrible.

### Peek at the learned embedding table

Recall that embedding table *is* the logit table, so we can read learned bigram structure straight out of it: index a character's row, softmax it into $P(\text{next char} \mid \text{this char})$, list the top successors. **Fill in the two marked lines of `top_next`.**

In [ ]:
emb = model.token_embedding_table.weight
print("embedding / logit table shape:", tuple(emb.shape))

def top_next(ch, k=5):
    row = None    # TODO 1: the row of `emb` for character ch (stoi turns ch into a row index)
    probs = None  # TODO 2: normalise that row into probabilities
    vals, idx = torch.topk(probs, k)
    return [(itos[i.item()], round(v.item(), 3)) for v, i in zip(vals, idx)]

for ch in ['t', 'h', 'o', 'a', ' ']:
    print(f"after {ch!r:4} ->", top_next(ch))

Look at `'t'`: its top successor is `'h'` (*the*, *that*, *this*), which is a real English regularity recovered purely from counts. Try `'q'` or `'z'` (much rarer tokens) too.

## Recap

**You built today:**

- A **character tokeniser** — `encode` / `decode`.
- The **pipeline** — data tensor, split, `get_batch`, `estimate_loss`, `train_model`
- A **Bigram model** — `nn.Embedding(vocab, vocab)` 
- Trained generation of (bad) Shakespeare.

That is every piece of a real language model — tokeniser, embedding, loss, training loop, sampler. What the bigram lacks is **context**.

### Extensions (optional)

- In `generate`, replace the `torch.multinomial` sample with an argmax. Why does the output get stuck repeating itself?
- Write the reverse of `top_next`: for a given character, which characters most like to *precede* it?

### Next time — Lab 9

**Lecture 9, Thursday 13 August: Recurrent Neural Networks** — a model with *memory*, same corpus and pipeline. *Keep your bigram sample; we'll race it against the RNN.*

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — `encode` / `decode`**

One list comprehension each; the `list(map(...))` versions a chatbot may offer are identical in effect.

In [ ]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

**Solution — `get_batch`**

The `- block_size` inside `randint` keeps every chunk and its shifted target inside the data; forgetting `.to(device)` works on CPU and breaks the day you get a GPU.

In [ ]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

**Solution — the bigram language model (`forward` + `generate`)**

The flatten `.view(B * T, C)` is the only non-obvious move — `F.cross_entropy` treats every position of every sequence as an independent prediction. `logits[:, -1, :]` keeps just the last time step; a bigram never uses anything earlier.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)        # (B,T,vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

**Solution — `top_next`**

Row `stoi[ch]` of `emb` holds one logit per possible successor; `F.softmax(row, dim=-1)` normalises it into probabilities.

In [ ]:
emb = model.token_embedding_table.weight
print("embedding / logit table shape:", tuple(emb.shape))

def top_next(ch, k=5):
    row = emb[stoi[ch]]
    probs = F.softmax(row, dim=-1)
    vals, idx = torch.topk(probs, k)
    return [(itos[i.item()], round(v.item(), 3)) for v, i in zip(vals, idx)]

for ch in ['t', 'h', 'o', 'a', ' ']:
    print(f"after {ch!r:4} ->", top_next(ch))